# EduRecSys – Hybrid Recommendation System

This notebook implements a **Hybrid Recommendation System** that combines
content-based filtering and collaborative filtering.

Hybrid recommendation leverages:
- **Content similarity** to handle cold-start scenarios.
- **User behavior patterns** to improve personalization and discovery.

By combining both approaches, the system aims to provide more accurate and
robust recommendations than either method alone.


In [1]:
import pandas as pd
import numpy as np

In [2]:
content_en = pd.read_csv("/kaggle/input/edurecsys-processed-content/content_en.csv")
interactions = pd.read_csv("/kaggle/input/edurecsys-interactions/interactions.csv")

print(content_en.shape)
print(interactions.shape)


(3678, 7)
(5980, 3)


# Build User–Item Matrix

In [3]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="content_id",
    values="rating"
).fillna(0)

user_item_matrix.shape


(500, 2937)

# User Similarity (Cosine)

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.iloc[:5, :5]


user_id,user_1,user_10,user_100,user_101,user_102
user_id,,,,,
user_1,1.0,0.0,0.0,0.0,0.0
user_10,0.0,1.0,0.0,0.0,0.0
user_100,0.0,0.0,1.0,0.0,0.0
user_101,0.0,0.0,0.0,1.0,0.0
user_102,0.0,0.0,0.0,0.0,1.0


# Content-Based Similarity (TF-IDF)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

tfidf_matrix = tfidf.fit_transform(content_en["description"])
tfidf_matrix.shape


(3678, 4078)

In [8]:
content_similarity = cosine_similarity(tfidf_matrix)
content_similarity.shape

(3678, 3678)

# User Interaction History

In this step, we extract the interaction history of a target user.
This includes all courses the user has interacted with along with their ratings.

Understanding user preferences is essential for hybrid recommendation systems,
as it provides the basis for both collaborative filtering and content-based recommendations.


In [9]:
def get_user_history(user_id, interactions_df, content_df):
    """
    Returns courses interacted with by a given user, merged with course metadata.
    """
    user_history = interactions_df[interactions_df["user_id"] == user_id]
    
    user_history = user_history.merge(
        content_df,
        on="content_id",
        how="left"
    )
    
    return user_history.sort_values(by="rating", ascending=False)


In [10]:
get_user_history("user_1", interactions, content_en).head()

,user_id,content_id,rating,title,description,subject,level,content_duration,language
0,user_1,71912,5,Have Fun with Beginner Blues Piano,Have Fun with Beginner Blues Piano,Musical Instruments,All Levels,2.500000,en
2,user_1,200722,5,Jazz Guitar Reharmonization for Autumn Leaves,Jazz Guitar Reharmonization for Autumn Leaves,Musical Instruments,All Levels,0.616667,en
3,user_1,961996,5,Learn Reactivex From Ground Up,Learn Reactivex From Ground Up,Web Development,Intermediate Level,5.000000,en
10,user_1,125162,5,Create and Deploy a Web App in 3 Hours,Create and Deploy a Web App in 3 Hours,Web Development,Beginner Level,3.500000,en
6,user_1,647336,5,Country Guitar Fundamentals,Country Guitar Fundamentals,Musical Instruments,Intermediate Level,1.500000,en


# Collaborative Filtering Candidates

In this step, we generate recommendation candidates using user-based collaborative filtering.
We identify users with similar interaction patterns and collect courses they have rated highly.

Courses already seen by the target user are excluded to ensure novel recommendations.


In [11]:
def get_cf_candidates(
    user_id,
    interactions_df,
    user_similarity_df,
    top_users=5
):
    # Get most similar users (excluding the user himself)
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
        .head(top_users)
        .index
    )
    
    # Get interactions of similar users
    similar_users_interactions = interactions_df[
        interactions_df["user_id"].isin(similar_users)
    ]
    
    # Remove items already seen by the target user
    seen_items = interactions_df[
        interactions_df["user_id"] == user_id
    ]["content_id"].unique()
    
    candidates = similar_users_interactions[
        ~similar_users_interactions["content_id"].isin(seen_items)
    ]
    
    # Rank by average rating
    return (
        candidates.groupby("content_id")["rating"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )


In [12]:
cf_candidates = get_cf_candidates(
    "user_1",
    interactions,
    user_similarity_df,
    top_users=5
)

cf_candidates.head()


,content_id,rating
0,153926,5.0
1,286424,5.0
2,861566,5.0
3,672112,5.0
4,373636,5.0


# Content-Based Candidates

In this step, we generate recommendation candidates based on content similarity.
We use the user's highly rated courses as reference items and compute cosine similarity
between their TF-IDF representations and all other courses.

This approach captures the user's topical interests directly from course descriptions.


In [22]:
def get_user_content_candidates(
    liked_items,
    content_df,
    content_similarity,
    top_k=10
):
    candidate_scores = {}

    for item_id in liked_items:
        idx = content_df.index[content_df["content_id"] == item_id][0]
        similarities = content_similarity[idx]

        for i, score in enumerate(similarities):
            candidate_id = content_df.iloc[i]["content_id"]
            candidate_scores[candidate_id] = max(
                candidate_scores.get(candidate_id, 0),
                score
            )

    # remove already seen items
    for item in liked_items:
        candidate_scores.pop(item, None)

    return (
        pd.DataFrame(candidate_scores.items(), columns=["content_id", "score"])
        .sort_values("score", ascending=False)
        .head(top_k)
    )


In [23]:
content_candidates_user = get_user_content_candidates(
    liked_items,
    content_en,
    content_similarity,
    top_k=10
)

content_candidates_user


,content_id,score
2704,836044,0.703257
2726,863954,0.653634
2024,1048188,0.554996
2039,648506,0.527055
1934,1132694,0.523284
2259,765056,0.495523
2451,199500,0.415524
3343,92194,0.411523
2423,618334,0.356802
2430,618370,0.355055


# Normalization

In [17]:
cf_candidates["rating_norm"] = (
    cf_candidates["rating"] - cf_candidates["rating"].min()
) / (
    cf_candidates["rating"].max() - cf_candidates["rating"].min()
)

cf_candidates.head()

,content_id,rating,rating_norm
0,153926,5.0,1.0
1,286424,5.0,1.0
2,861566,5.0,1.0
3,672112,5.0,1.0
4,373636,5.0,1.0


In [26]:
content_candidates_user["score_norm"] = (
    content_candidates_user["score"] - content_candidates_user["score"].min()
) / (
    content_candidates_user["score"].max() - content_candidates_user["score"].min()
)

content_candidates_user.head()

,content_id,score,score_norm
2704,836044,0.703257,1.000000
2726,863954,0.653634,0.857489
2024,1048188,0.554996,0.574209
2039,648506,0.527055,0.493967
1934,1132694,0.523284,0.483139


## Score Normalization

Since collaborative filtering ratings and content-based similarity scores are on different scales,
both values were normalized to the range [0, 1].

This ensures fair contribution from each recommendation component when computing the final hybrid score.


# Hybrid Scoring

In [28]:
hybrid_df = pd.merge(
    cf_candidates[["content_id", "rating_norm"]],
    content_candidates_user[["content_id", "score_norm"]],
    on="content_id",
    how="outer"
).fillna(0)

hybrid_df.head()


,content_id,rating_norm,score_norm
0,13216,0.5,0.00000
1,92194,0.0,0.16217
2,153926,1.0,0.00000
3,158934,0.0,0.00000
4,199450,0.0,0.00000


In [31]:
alpha = 0.6  # CF weight
beta = 0.4   # Content-based weight

hybrid_df["hybrid_score"] = (
    alpha * hybrid_df["rating_norm"] +
    beta * hybrid_df["score_norm"]
)

hybrid_df.sort_values("hybrid_score", ascending=False).head()


,content_id,rating_norm,score_norm,hybrid_score
2,153926,1.0,0.0,0.6
7,286424,1.0,0.0,0.6
13,373636,1.0,0.0,0.6
14,415802,1.0,0.0,0.6
35,861566,1.0,0.0,0.6


The weights α and β control the contribution of collaborative filtering
and content-based similarity respectively. In this experiment, a higher
weight is assigned to collaborative filtering to prioritize behavioral signals.


## Observation

The current hybrid results are dominated by collaborative filtering signals.
This occurs because top collaborative candidates do not strongly overlap
with content-based similarity results.

This behavior is expected in sparse interaction settings and highlights
the complementary nature of both approaches.


# Final Hybrid Recommendation

In this final step, collaborative filtering and content-based similarity are combined into a single hybrid score.
The hybrid score balances user behavioral patterns with content relevance, resulting in more personalized and robust recommendations.

In [32]:
final_recommendations = (
    hybrid_df
    .sort_values("hybrid_score", ascending=False)
    .head(10)
    .merge(content_en, on="content_id", how="left")
)

final_recommendations[
    ["content_id", "title", "subject", "level", "hybrid_score"]
]


,content_id,title,subject,level,hybrid_score
0,153926,BITCOIN VISUALLY. Part I.,Business Finance,Beginner Level,0.600000
1,286424,Learn to Design a Logo in Adobe Illustrator,Graphic Design,All Levels,0.600000
2,373636,Master the Trombone: Intermediate Instruction ...,Musical Instruments,All Levels,0.600000
3,415802,Accounting Standards Basics (Professional Cour...,Business Finance,All Levels,0.600000
4,861566,Instant harmonica - play Adele's wonderful 'He...,Musical Instruments,All Levels,0.600000
5,856968,How to Make a Wordpress Website 2017,Web Development,Beginner Level,0.600000
6,1076222,The most popular techniques in Photoshop,Graphic Design,Intermediate Level,0.600000
7,672112,誰でもわかる Adobe Illustrator CS5,Graphic Design,Beginner Level,0.600000
8,836044,React From The Ground Up,Web Development,All Levels,0.400000
9,863954,Angular 2 From The Ground Up,Web Development,All Levels,0.342996


In [33]:
import joblib

joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(content_similarity, "content_similarity.pkl")


['content_similarity.pkl']